In [0]:
import pandas as pd

# Baixa a tabela de municípios do IBGE (mesma fonte já validada antes)
municipios = pd.read_csv("https://raw.githubusercontent.com/kelvins/municipios-brasileiros/main/csv/municipios.csv")
municipios["codigo_municipio"] = municipios["codigo_ibge"] // 10  # remove o dígito verificador

dim_uf = pd.DataFrame([
    (11, "RO", "Rondônia", "Norte"), (12, "AC", "Acre", "Norte"), (13, "AM", "Amazonas", "Norte"),
    (14, "RR", "Roraima", "Norte"), (15, "PA", "Pará", "Norte"), (16, "AP", "Amapá", "Norte"), (17, "TO", "Tocantins", "Norte"),
    (21, "MA", "Maranhão", "Nordeste"), (22, "PI", "Piauí", "Nordeste"), (23, "CE", "Ceará", "Nordeste"),
    (24, "RN", "Rio Grande do Norte", "Nordeste"), (25, "PB", "Paraíba", "Nordeste"), (26, "PE", "Pernambuco", "Nordeste"),
    (27, "AL", "Alagoas", "Nordeste"), (28, "SE", "Sergipe", "Nordeste"), (29, "BA", "Bahia", "Nordeste"),
    (31, "MG", "Minas Gerais", "Sudeste"), (32, "ES", "Espírito Santo", "Sudeste"), (33, "RJ", "Rio de Janeiro", "Sudeste"),
    (35, "SP", "São Paulo", "Sudeste"),
    (41, "PR", "Paraná", "Sul"), (42, "SC", "Santa Catarina", "Sul"), (43, "RS", "Rio Grande do Sul", "Sul"),
    (50, "MS", "Mato Grosso do Sul", "Centro-Oeste"), (51, "MT", "Mato Grosso", "Centro-Oeste"),
    (52, "GO", "Goiás", "Centro-Oeste"), (53, "DF", "Distrito Federal", "Centro-Oeste"),
], columns=["codigo_uf", "sigla_uf", "nome_uf", "regiao"])

# Junta município com sua UF/região
dim_municipio = municipios.merge(dim_uf, on="codigo_uf")[
    ["codigo_municipio", "nome", "codigo_uf", "sigla_uf", "nome_uf", "regiao", "latitude", "longitude"]
].rename(columns={"nome": "nome_municipio"})

# Cria as 27 linhas sintéticas de "Ignorado" (uma por UF, código UFxx0000,
# o padrão que achamos nos 2.234 registros órfãos lá na exploração)
ignorados = dim_uf.copy()
ignorados["codigo_municipio"] = ignorados["codigo_uf"] * 10000
ignorados["nome_municipio"] = "Ignorado (" + ignorados["sigla_uf"] + ")"
ignorados["latitude"] = None
ignorados["longitude"] = None
ignorados = ignorados[["codigo_municipio", "nome_municipio", "codigo_uf", "sigla_uf", "nome_uf", "regiao", "latitude", "longitude"]]

# Junta tudo: municípios reais + municípios "ignorado" sintéticos
dim_municipio_final = pd.concat([dim_municipio, ignorados], ignore_index=True)
print("Total de linhas em dim_municipio (esperado: 5.571 + 27 = 5.598):", len(dim_municipio_final))

# Grava como tabela do Databricks
spark_dim_municipio = spark.createDataFrame(dim_municipio_final)
(spark_dim_municipio.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_dim_municipio"))

print("Tabela 'gold_dim_municipio' gravada.")